In [1]:
# Import libraries required for feature engineering

import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Load the cleaned pricing dataset

DATA_PATH = Path("../data/processed/pricing_data.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

Dataset shape: (13774324, 20)
Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00


In [3]:
# Create calendar features for weekly seasonality and weekend demand patterns

df["week_of_year"] = df["date"].dt.isocalendar().week.astype("int16")

df["is_weekend"] = (
    df["weekday"].isin(["Saturday", "Sunday"])
).astype("int8")

print(
    df[
        ["date", "weekday", "wday", "month", "year",
         "week_of_year", "is_weekend"]
    ].head()
)

        date   weekday  wday  month  year  week_of_year  is_weekend
0 2011-01-29  Saturday     1      1  2011             4           1
1 2011-01-29  Saturday     1      1  2011             4           1
2 2011-01-29  Saturday     1      1  2011             4           1
3 2011-01-29  Saturday     1      1  2011             4           1
4 2011-01-29  Saturday     1      1  2011             4           1


In [4]:
# Create one SNAP indicator corresponding to the state of each store

df["snap_active"] = np.select(
    [
        df["state_id"] == "CA",
        df["state_id"] == "TX",
        df["state_id"] == "WI"
    ],
    [
        df["snap_CA"],
        df["snap_TX"],
        df["snap_WI"]
    ]
).astype("int8")

print(df["snap_active"].value_counts().sort_index())

snap_active
0    9237090
1    4537234
Name: count, dtype: int64


In [6]:
# Create lag and rolling demand features using only past demand to avoid data leakage

df = df.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

group_cols = ["item_id", "store_id"]

df["lag_1_demand"] = (
    df.groupby(group_cols, observed=True)["demand"]
    .shift(1)
)

df["lag_7_demand"] = (
    df.groupby(group_cols, observed=True)["demand"]
    .shift(7)
)

df["rolling_7d_demand"] = (
    df.groupby(group_cols, observed=True)["demand"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

df["rolling_28d_demand"] = (
    df.groupby(group_cols, observed=True)["demand"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

display(
    df[
        [
            "date", "item_id", "store_id", "demand",
            "lag_1_demand", "lag_7_demand",
            "rolling_7d_demand", "rolling_28d_demand"
        ]
    ].head(35)
)

,date,item_id,store_id,demand,lag_1_demand,lag_7_demand,rolling_7d_demand,rolling_28d_demand
0,2011-01-29,FOODS_1_001,CA_3,1,NaN,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,CA_3,2,1.0,NaN,NaN,NaN
2,2011-01-31,FOODS_1_001,CA_3,1,2.0,NaN,NaN,NaN
3,2011-02-01,FOODS_1_001,CA_3,1,1.0,NaN,NaN,NaN
4,2011-02-02,FOODS_1_001,CA_3,1,1.0,NaN,NaN,NaN
5,2011-02-03,FOODS_1_001,CA_3,2,1.0,NaN,NaN,NaN
6,2011-02-04,FOODS_1_001,CA_3,0,2.0,NaN,NaN,NaN
7,2011-02-05,FOODS_1_001,CA_3,1,0.0,1.0,1.142857,NaN
8,2011-02-06,FOODS_1_001,CA_3,1,1.0,2.0,1.142857,NaN
9,2011-02-07,FOODS_1_001,CA_3,1,1.0,1.0,1.000000,NaN


In [8]:
# Create historical price features for measuring changes in selling price

price_group = df.groupby(
    ["item_id", "store_id"],
    observed=True
)["sell_price"]

df["previous_price"] = price_group.shift(1)

df["price_change_pct"] = (
    (df["sell_price"] - df["previous_price"])
    / df["previous_price"]
) * 100

df["price_changed"] = (
    df["previous_price"].notna() &
    (df["sell_price"] != df["previous_price"])
).astype("int8")

display(
    df[
        [
            "date", "item_id", "store_id",
            "previous_price", "sell_price",
            "price_change_pct", "price_changed"
        ]
    ].head(35)
)

,date,item_id,store_id,previous_price,sell_price,price_change_pct,price_changed
0,2011-01-29,FOODS_1_001,CA_3,NaN,2.0,NaN,0
1,2011-01-30,FOODS_1_001,CA_3,2.0,2.0,0.0,0
2,2011-01-31,FOODS_1_001,CA_3,2.0,2.0,0.0,0
3,2011-02-01,FOODS_1_001,CA_3,2.0,2.0,0.0,0
4,2011-02-02,FOODS_1_001,CA_3,2.0,2.0,0.0,0
5,2011-02-03,FOODS_1_001,CA_3,2.0,2.0,0.0,0
6,2011-02-04,FOODS_1_001,CA_3,2.0,2.0,0.0,0
7,2011-02-05,FOODS_1_001,CA_3,2.0,2.0,0.0,0
8,2011-02-06,FOODS_1_001,CA_3,2.0,2.0,0.0,0
9,2011-02-07,FOODS_1_001,CA_3,2.0,2.0,0.0,0


In [9]:
# Remove initial rows that do not have enough historical demand for model features

required_history_features = [
    "lag_1_demand",
    "lag_7_demand",
    "rolling_7d_demand",
    "rolling_28d_demand"
]

rows_before = len(df)

df = df.dropna(
    subset=required_history_features
).reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after:", len(df))
print("Rows removed:", rows_before - len(df))

print("\nMissing historical features:")
print(df[required_history_features].isna().sum())

Rows before: 13774324
Rows after: 13563568
Rows removed: 210756

Missing historical features:
lag_1_demand          0
lag_7_demand          0
rolling_7d_demand     0
rolling_28d_demand    0
dtype: int64


In [10]:
# Validate the engineered features before saving the modeling dataset

engineered_features = [
    "sell_price",
    "week_of_year",
    "is_weekend",
    "snap_active",
    "lag_1_demand",
    "lag_7_demand",
    "rolling_7d_demand",
    "rolling_28d_demand",
    "previous_price",
    "price_change_pct",
    "price_changed"
]

print("Final shape:", df.shape)

print("\nMissing values:")
print(df[engineered_features].isna().sum())

print("\nInfinite values:")
print(
    np.isinf(
        df[engineered_features].select_dtypes(include=np.number)
    ).sum()
)

Final shape: (13563568, 30)

Missing values:
sell_price            0
week_of_year          0
is_weekend            0
snap_active           0
lag_1_demand          0
lag_7_demand          0
rolling_7d_demand     0
rolling_28d_demand    0
previous_price        0
price_change_pct      0
price_changed         0
dtype: int64

Infinite values:
sell_price            0
week_of_year          0
is_weekend            0
snap_active           0
lag_1_demand          0
lag_7_demand          0
rolling_7d_demand     0
rolling_28d_demand    0
previous_price        0
price_change_pct      0
price_changed         0
dtype: int64


In [11]:
# Save the feature-engineered dataset for elasticity analysis and demand modeling

OUTPUT_PATH = Path("../data/processed/pricing_features.parquet")

df.to_parquet(
    OUTPUT_PATH,
    index=False,
    engine="pyarrow"
)

print("Saved to:", OUTPUT_PATH)
print("Final rows:", len(df))
print("Final columns:", df.shape[1])
print(f"File size: {OUTPUT_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Saved to: ..\data\processed\pricing_features.parquet
Final rows: 13563568
Final columns: 30
File size: 77.34 MB
